_This notebook was developed by [Keneth Garcia](https://www.linkedin.com/in/keneth-garcia-a6305b1b9/). Source and license info are on [GitHub](https://github.com/KenethGarcia/SieveComP)._

In [1]:
import pandas as pd
from sievecomp.io.sql import load_sql
from sievecomp.core.runner import Runner

# **_SieveComP_**: Advanced Usage

For those user that do not want to rely on the default TOML structure of **_SieveComP_**, this notebook will show how to use the library in a more advanced way, without requiring the user to create a TOML file. This is useful for those that want to use **_SieveComP_** in a more programmatic way, or for those that want to integrate **_SieveComP_** into their own codebase.

As a baseline of comparison, we will state some examples where I will show both the TOML and the programmatic way of using **_SieveComP_**. The examples stated here are real uses cases that I have encountered in my work, and I hope they will be useful for you as well.

One of **_SieveComP_**'s most powerful features is its ability to bridge static configurations with dynamic Python applications. While your TOML files define standardized, reproducible SQL queries, building custom interfaces (like a CLI or GUI) often requires filtering data on the fly based on user input, such as date ranges.


## Building Dynamic SeisComP Queries with the Runner Class

As we already show in the duplicates tutorial notebook, instead of hardcoding new SQL files for every possible filter combination, you can use a "query wrapping" technique. By loading the static SQL from your configuration and wrapping it inside a dynamic subquery, you can safely inject new parameters without breaking the original SQL structure.

This is really useful when developing custom applications on top of **_SieveComP_**, given that you often need to fetch raw catalog data for a specific time window before deciding whether to run the full quality-control engine. To achieve this, we can write a wrapper function that intercepts the SQL query defined in your TOML file, injects dynamic `WHERE` clauses for the start and end times, and executes it using the `Runner`'s override parameters.

> NOTE: In static cases where you don't need to create a lot of new SQL or TOML files, you can just use the `Runner` class with a full query including the time range to make this easier (see the Quickstart guide).

In [2]:
def fetch_events_by_time_range(
        runner: Runner,
        query_name: str,
        start_time: pd.Timestamp,
        end_time: pd.Timestamp
) -> pd.DataFrame:
    """
    Fetches events from a specific SeisComP database query within a dynamic time window.
    Bypasses the engine's checks and duplicates logic.
    """
    # 1. Retrieve the configured time column used for sorting and filtering
    time_col = runner._cm.get_time_column()

    # 2. Prepare the dynamic time bounds and SQL parameters
    sql_params = {}
    where_clauses = []

    if start_time:
        sql_params["start_time"] = start_time.strftime('%Y-%m-%d %H:%M:%S')
        where_clauses.append(f"base_query.{time_col} >= %(start_time)s")

    if end_time:
        sql_params["end_time"] = end_time.strftime('%Y-%m-%d %H:%M:%S')
        where_clauses.append(f"base_query.{time_col} < %(end_time)s")

    where_sql = " AND ".join(where_clauses)

    # 3. Load the base SQL text strictly as it is defined in the TOML configuration
    query_cfg = runner._cm.select_query(name=query_name)
    base_sql = load_sql(
        query_cfg=query_cfg,
        base_dir=runner._cm.base_dir,
        config_name=runner._cm.config_path.name
    )

    # 4. Wrap the base query as a subquery, applying the new WHERE clauses and sorting
    wrapped_sql = f"""SELECT * FROM ({base_sql}) AS base_query WHERE {where_sql} ORDER BY base_query.{time_col} ASC"""

    # 5. Fetch the events (skipping the check/duplicate evaluation steps for raw data retrieval)
    result = runner.run(
        query_name=query_name,
        multi_query=False,
        perform_duplicates=False,
        perform_checks=False,
        sql_text_override=wrapped_sql,
        sql_params=sql_params
    )

    # 6. Return the raw queried events DataFrame
    return result.total_events

**How the Wrapper Works:**
- Config Extraction (Steps 1 & 3): We use the `Runner`'s internal Configuration Manager (`_cm`) to extract the exact name of the time column and the raw SQL string from your existing TOML definitions. This ensures your script remains synchronized with your configuration file.

- Safe Parameterization (Step 2): We map the pandas Timestamps into standard SQL datetime strings and load them into a dictionary (`sql_params`). **_SieveComP_**'s engine passes these to the database driver safely, preventing SQL injection vulnerabilities.

- The Subquery Trick (Step 4): Simply appending `WHERE start_time = X` to a complex SQL file full of `JOIN` statements will usually cause syntax errors. By placing the original query inside `FROM (...) AS base_query`, we create a clean, safe table representation that we can easily filter and sort.

- Engine Overrides (Step 5): The `runner.run()` method accepts a `sql_text_override`. By setting `perform_duplicates=False` and `perform_checks=False`, we tell the **_SieveComP_** engine to act purely as a data fetcher, returning the unflagged data instantly.

**Usage Example:**

Once defined, you can use this function to easily paginate through days or months of data (don't forget to set the connection parameters in your TOML file):

In [3]:
# Initialize the runner
runner = Runner(config_path='../../examples/data/configs/seismic_revision_routine.toml')

start = pd.to_datetime("2026-04-01 00:00:00", utc=True)
end = pd.to_datetime("2026-04-02 00:00:00", utc=True)

# Fetch just the events for the specified window
events_df = fetch_events_by_time_range(
    runner=runner,
    query_name="SeisComP6",
    start_time=start,
    end_time=end
)

print("Number of events by event_type:")
print(events_df['event_type'].value_counts())

Number of events by event_type:
event_type
not locatable                  116
earthquake                      90
outside of network interest      5
not existing                     2
explosion                        2
Name: count, dtype: int64


# Beyond the TOML: Custom Workflows and Advanced Architecture

While **_SieveComP_**'s TOML files excel at creating standardized, reproducible quality-control pipelines, building custom tools—such as an interactive Jupyter dashboard, a Streamlit web app, or a specialized data parser—demands greater flexibility. You often need to filter data on the fly based on user input, tweak checking parameters without editing text files, or simply process raw data at maximum speed.

To bridge the gap between static configurations and dynamic Python applications, **_SieveComP_** is built with a layered architecture. Depending on your specific use case, you can interact with the engine at three distinct depths:

1. **The Dynamic Wrapper (High-Level):** As we explained previously, the dynamic wrapper allows you to intercept static SQL queries from your configuration and wrap them in dynamic bounds to fetch specific data slices directly from the database.

2. **In-Memory Rule Injection (Mid-Level):** Bypassing the `.toml` file entirely by constructing programmatic condition dictionaries and pushing them directly into the `Runner`'s active memory for instant re-evaluations.

3. **Raw Vectorized Masking (Low-Level):** Stripping away the configuration manager completely to apply **_SieveComP_**'s lightning-fast, core NumPy masking functions directly to your Pandas DataFrames.

The following examples will break down how to implement the last two approaches, giving you complete programmatic control over how you query, evaluate, and filter your seismic catalogs.

## 1. In-Memory Rule Injection - High RMS Events

Let's suppose that you want to filter out events with a high RMS value, but you don't want to create a new TOML file for this. Instead, you can define the condition in memory and inject it into the `Runner`'s active configuration.

Because **_SieveComP_** parses TOML files directly into Python dictionaries under the hood, creating a programmatic instance of a TOML configuration simply means constructing the equivalent Python dictionary and injecting it into the `Runner` class:

In [6]:
# Check for events with a high RMS value (quality_standardError > 1.50)
high_RMS_rules = [
    {
        "name": "High RMS",
        "event_type": "earthquake",
        "conditions": [
            {
                "rule_type": "numeric",
                "column": "quality_standardError",
                "mode": "gt",
                "threshold": 1.0
            }
        ]
    }
]

Once you have your programmatic dictionary, you can bypass the `.toml` file entirely by modifying the ConfigManager's data directly in memory.

In [8]:
# 1. Initialize the runner with a base configuration (for DB connection, etc.)
runner = runner

# 2. We will use the cached events DataFrame from the previous example, but you can also fetch new events using the dynamic wrapper function.
raw_events = events_df

# 3. Define the programmatic check (stated above)
high_RMS_rules = high_RMS_rules

# 4. Inject the custom check directly into the Runner's configuration memory
runner._cm.config_data["checks"] = high_RMS_rules

# 5. Run the engine using the cached data
# CRITICAL: set reload_config=False so the Runner uses our injected dictionary instead of reading the TOML file again!
result = runner.update_from_cache(
    events_df=raw_events,
    reload_config=False
)

print(f"Events matching the programmatic High RMS check: {len(result.checks)}")

Events matching the programmatic High RMS check: 14


## 2. Raw Vectorized Masking - Invalid Labeling

Let's suppose you want to filter out events that have invalid labeling, such as events with a `label` value of `None` or an empty string. Instead of creating a new TOML file or injecting rules into the Runner, you can directly apply vectorized masking using **_SieveComP_**'s core functions. This approach is the fastest and most flexible, allowing you to manipulate your DataFrame directly.

Translating a categorical TOML check into a raw vectorized mask requires mapping the keys from the [[checks.conditions]] block:

```toml
[[checks]]
name = "Invalid label"

    [[checks.conditions]]
    rule_type = "category"
    column = "event_type"
    mode = "not_in"
    values = ["earthquake", "not locatable", "volcanic eruption", "explosion", "not existing", "outside of network interest"]
```

directly to the arguments of the available functions at `sievecomp.checks.vectorized_masks`. The following example shows how to implement this check programmatically:

In [10]:
from sievecomp.checks.vectorized_masks import non_numeric_mask

valid_labels = [
    "earthquake",
    "not locatable",
    "volcanic eruption",
    "explosion",
    "not existing",
    "outside of network interest"
]

# Translate the TOML condition directly into the vectorized function
invalid_label_mask = non_numeric_mask(
    events=events_df,
    column="event_type",
    mode="not_in",
    values=valid_labels
)

# Apply the boolean mask to isolate events with invalid labels
flagged_events = events_df[invalid_label_mask]

print(f"Events with invalid types found: {len(flagged_events)}")

Events with invalid types found: 0
